<a href="https://colab.research.google.com/github/sahmedshereen-prog/Customer-Churn-Prediction/blob/main/Tsk_2_Uneeq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Sentiment Analysis on Social Media Data
=========================================
Task: Perform sentiment analysis on a dataset of social media posts/reviews
using NLP techniques.

Data files used (place both next to this script):
- Reddit_Data.csv   (columns: clean_comment, category)
- Twitter_Data.csv  (columns: clean_text, category)

category values: -1 = negative, 0 = neutral, 1 = positive

Why combine both:
Using both sources gives a larger, more diverse training set (different
writing styles: Reddit comments vs tweets), which usually makes the model
more robust than training on just one source.

What this script does:
1. Loads both CSVs and unifies them into one dataframe (text, category)
2. Cleans the text (handles NaNs, removes duplicates, basic text cleaning)
3. Preprocesses text with NLP: lowercasing, stopword removal, lemmatization
4. Converts text to numeric features using TF-IDF
5. Trains multiple classification models
6. Evaluates using accuracy, precision, recall, F1-score per class
   (positive / neutral / negative), since classes are usually imbalanced
7. Saves the best model + vectorizer for reuse
"""

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import joblib

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

reddit_df = pd.read_csv("Reddit_Data.csv")
twitter_df = pd.read_csv("Twitter_Data.csv")

reddit_df = reddit_df.rename(columns={"clean_comment": "text"})
twitter_df = twitter_df.rename(columns={"clean_text": "text"})

df = pd.concat([reddit_df[["text", "category"]],
                 twitter_df[["text", "category"]]], ignore_index=True)

print("Combined shape:", df.shape)

df.dropna(subset=["text", "category"], inplace=True)
df.drop_duplicates(subset=["text"], inplace=True)
df["category"] = df["category"].astype(int)

label_map = {-1: "negative", 0: "neutral", 1: "positive"}
df["sentiment"] = df["category"].map(label_map)

print("\nClass balance:")
print(df["sentiment"].value_counts(normalize=True))

plt.figure(figsize=(5, 4))
sns.countplot(x="sentiment", data=df, order=["negative", "neutral", "positive"])
plt.title("Sentiment Class Distribution")
plt.savefig("sentiment_distribution.png", bbox_inches="tight")
plt.close()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in STOPWORDS and len(w) > 2]
    return " ".join(tokens)

print("\nCleaning text (this may take a minute)...")
df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0]

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"],
    test_size=0.2, random_state=42, stratify=df["sentiment"]
)

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(class_weight="balanced", max_iter=5000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
}

results = {}

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)

    results[name] = {
        "accuracy": acc,
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
    }

    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred, labels=["negative", "neutral", "positive"]))

results_df = pd.DataFrame(results).T.sort_values("macro_f1", ascending=False)
print("\n=== Model comparison (sorted by macro F1) ===")
print(results_df)

results_df.to_csv("model_comparison_results.csv")

best_model_name = results_df.index[0]
best_model = models[best_model_name]
print(f"\nBest model: {best_model_name}")

joblib.dump(best_model, "best_sentiment_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("\nDone. Outputs saved: best_sentiment_model.pkl, tfidf_vectorizer.pkl, "
      "model_comparison_results.csv, sentiment_distribution.png")

def predict_sentiment(text):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    return best_model.predict(vec)[0]

sample = "This product is amazing, I really love it!"
print(f"\nSample prediction for: '{sample}'")
print("Predicted sentiment:", predict_sentiment(sample))

Combined shape: (200229, 2)

Class balance:
sentiment
positive    0.440603
neutral     0.340302
negative    0.219095
Name: proportion, dtype: float64

Cleaning text (this may take a minute)...

=== Logistic Regression ===
              precision    recall  f1-score   support

    negative       0.80      0.84      0.82      8751
     neutral       0.86      0.95      0.91     13542
    positive       0.94      0.85      0.89     17597

    accuracy                           0.88     39890
   macro avg       0.87      0.88      0.87     39890
weighted avg       0.89      0.88      0.88     39890

Confusion Matrix:
 [[ 7347   717   687]
 [  428 12904   210]
 [ 1384  1311 14902]]

=== Naive Bayes ===
              precision    recall  f1-score   support

    negative       0.83      0.45      0.59      8751
     neutral       0.80      0.67      0.73     13542
    positive       0.66      0.90      0.76     17597

    accuracy                           0.72     39890
   macro avg       0.

KeyboardInterrupt: 